# Đánh giá độ ổn định của các mô hình phân cụm

Notebook này lấy mẫu lại bộ RFM đã chuẩn hóa nhiều lần, huấn luyện lại ba mô hình
tốt nhất của Task 12 trên từng tập mẫu và đo độ giống nhau giữa các kết quả bằng
Adjusted Rand Index.

In [1]:
import sys
import warnings

sys.path.append('../src')

warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
from models.clustering import load_scaled_rfm
from evaluation.stability import (
    N_ITERATIONS,
    SAMPLE_SIZE,
    RANDOM_STATE,
    load_best_configs,
    generate_samples,
    run_resampling,
    pairwise_ari,
    reference_ari,
    summarize_stability,
    save_results,
)

## 1. Chuẩn bị dữ liệu đầu vào

Đọc bộ RFM đã chuẩn hóa và tải cấu hình tối ưu của từng thuật toán từ kết quả
thực nghiệm của Task 12 (`outputs/results/clustering_experiments.csv`), chọn theo
Silhouette Score cao nhất của mỗi thuật toán.

In [2]:
df, X = load_scaled_rfm('../data/processed/rfm_scaled.csv')
df.head()

Loaded ../data/processed/rfm_scaled.csv
  Customers: 4324
  Features : ['Recency', 'Frequency', 'Monetary']
  Missing  : {'Recency': 0, 'Frequency': 0, 'Monetary': 0}
  Dtypes   : {'Recency': dtype('float64'), 'Frequency': dtype('float64'), 'Monetary': dtype('float64')}


,CustomerID,Recency,Frequency,Monetary
0,12347.0,-0.901281,1.088034,1.445284
1,12348.0,-0.173081,0.396756,0.567579
2,12349.0,-0.741675,-0.950919,0.578791
3,12350.0,2.171126,-0.950919,-0.697983
4,12352.0,-0.562119,0.891637,0.465850


In [3]:
configs = load_best_configs('../outputs/results/clustering_experiments.csv')
configs

Loaded best configurations from ../outputs/results/clustering_experiments.csv (metric: silhouette)
  GMM     : {'n_components': 3, 'covariance_type': 'spherical'}
  HDBSCAN : {'min_cluster_size': 50, 'min_samples': None}
  K-Means : {'n_clusters': 3}


{'GMM': {'n_components': 3, 'covariance_type': 'spherical'},
 'HDBSCAN': {'min_cluster_size': 50, 'min_samples': None},
 'K-Means': {'n_clusters': 3}}

## 2. Xây dựng quy trình lấy mẫu lại dữ liệu

Mỗi lần lặp lấy ngẫu nhiên không hoàn lại 80% số khách hàng. Toàn bộ 50 tập mẫu
được sinh từ một `random_state` duy nhất nên quy trình tái lập được, và ba thuật
toán dùng chung đúng 50 tập mẫu đó để việc so sánh là công bằng.

In [4]:
samples = generate_samples(len(X), N_ITERATIONS, SAMPLE_SIZE, RANDOM_STATE)

print(f'Số lần lặp        : {N_ITERATIONS}')
print(f'Tỷ lệ lấy mẫu     : {SAMPLE_SIZE:.0%}')
print(f'Kích thước mỗi tập: {len(samples[0])} / {len(X)} khách hàng')
overlap = [len(np.intersect1d(samples[i], samples[j]))
           for i in range(N_ITERATIONS) for j in range(i + 1, N_ITERATIONS)]
print(f'Số cặp lần lặp    : {len(overlap)}')
print(f'Số khách hàng chung trung bình mỗi cặp: {np.mean(overlap):.0f} '
      f'({np.mean(overlap) / len(samples[0]):.1%} của một tập mẫu)')

Số lần lặp        : 50
Tỷ lệ lấy mẫu     : 80%
Kích thước mỗi tập: 3459 / 4324 khách hàng


Số cặp lần lặp    : 1225
Số khách hàng chung trung bình mỗi cặp: 2767 (80.0% của một tập mẫu)


## 3. Huấn luyện lại các thuật toán phân cụm

`run_resampling` huấn luyện lại cả ba thuật toán trên từng tập mẫu và ghi lại
nhãn cụm của mỗi lần lặp vào một ma trận `(50, 4324)`. Các ô ứng với khách hàng
không nằm trong tập mẫu của lần lặp đó được đánh dấu riêng để bước tính ARI chỉ
so sánh trên phần dữ liệu chung của từng cặp.

Mỗi thuật toán còn ghi lại thông tin riêng: K-Means ghi số cụm và inertia, GMM ghi
xác suất thuộc cụm và log-likelihood, HDBSCAN ghi số cụm phát hiện được và số
điểm nhiễu.

In [5]:
label_matrices, iterations = run_resampling(X, configs, N_ITERATIONS, SAMPLE_SIZE, RANDOM_STATE)

print()
print('Kích thước ma trận nhãn:', {model: matrix.shape for model, matrix in label_matrices.items()})
iterations.head()

  GMM     : trained on 50 resampled datasets


  HDBSCAN : trained on 50 resampled datasets
  K-Means : trained on 50 resampled datasets

Kích thước ma trận nhãn: {'GMM': (50, 4324), 'HDBSCAN': (50, 4324), 'K-Means': (50, 4324)}


,model,iteration,n_sample,n_clusters,train_time,log_likelihood,mean_max_proba,low_confidence_ratio,n_noise,noise_ratio,inertia
0,GMM,0,3459,3,0.026117,-12138.977766,0.927216,0.144550,NaN,NaN,NaN
1,GMM,1,3459,3,0.005720,-12105.785085,0.929175,0.139925,NaN,NaN,NaN
2,GMM,2,3459,3,0.007030,-12235.726555,0.927392,0.144261,NaN,NaN,NaN
3,GMM,3,3459,3,0.004439,-12181.601936,0.929722,0.140503,NaN,NaN,NaN
4,GMM,4,3459,3,0.004291,-12291.864734,0.926993,0.145418,NaN,NaN,NaN


### 3.1. K-Means

Ghi lại số cụm tạo thành và inertia của từng lần lặp.

In [6]:
kmeans_detail = iterations[iterations['model'] == 'K-Means']
kmeans_detail[['n_clusters', 'inertia', 'train_time']].describe().T

,count,mean,std,min,25%,50%,75%,max
n_clusters,50.0,3.000000,0.000000,3.000000,3.000000,3.000000,3.000000,3.000000
inertia,50.0,3420.578644,46.491664,3336.534387,3376.196879,3417.741897,3454.068244,3520.114776
train_time,50.0,0.002888,0.000620,0.001694,0.002468,0.002880,0.003379,0.004480


### 3.2. GMM

Ghi lại số cụm, log-likelihood và xác suất thuộc cụm. `mean_max_proba` là xác suất
trung bình của cụm được gán, `low_confidence_ratio` là tỷ lệ khách hàng có xác
suất cao nhất dưới 0,8.

In [7]:
gmm_detail = iterations[iterations['model'] == 'GMM']
gmm_detail[['n_clusters', 'log_likelihood', 'mean_max_proba', 'low_confidence_ratio']].describe().T

,count,mean,std,min,25%,50%,75%,max
n_clusters,50.0,3.000000,0.000000,3.000000,3.000000,3.000000,3.000000,3.000000
log_likelihood,50.0,-12203.094111,60.740741,-12343.398705,-12249.880478,-12200.740090,-12145.129867,-12105.785085
mean_max_proba,50.0,0.928002,0.001445,0.925004,0.927006,0.928075,0.928982,0.931622
low_confidence_ratio,50.0,0.143041,0.003609,0.133276,0.140214,0.143683,0.145635,0.149465


### 3.3. HDBSCAN

Ghi lại số cụm phát hiện được và số điểm nhiễu của từng lần lặp.

In [8]:
hdbscan_detail = iterations[iterations['model'] == 'HDBSCAN']
print(hdbscan_detail['n_clusters'].value_counts().rename('số lần lặp').to_string())
hdbscan_detail[['n_clusters', 'n_noise', 'noise_ratio']].describe().T

n_clusters
3    49
4     1


,count,mean,std,min,25%,50%,75%,max
n_clusters,50.0,3.020000,0.141421,3.000000,3.000000,3.000000,3.000000,4.000000
n_noise,50.0,839.680000,43.717292,797.000000,824.000000,834.500000,847.250000,1117.000000
noise_ratio,50.0,0.242752,0.012639,0.230413,0.238219,0.241255,0.244941,0.322926


## 4. Đánh giá độ ổn định bằng Adjusted Rand Index

ARI được tính giữa mọi cặp trong 50 lần lặp, tức 1.225 cặp cho mỗi thuật toán.
Với mỗi cặp, chỉ những khách hàng xuất hiện ở cả hai tập mẫu mới được đưa vào
phép tính, vì ARI đòi hỏi hai nhãn trên cùng một tập điểm.

In [9]:
ari_scores = {model: pairwise_ari(matrix) for model, matrix in label_matrices.items()}

pd.DataFrame({
    model: pd.Series(scores).describe()
    for model, scores in ari_scores.items()
}).T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

,count,mean,std,min,25%,50%,75%,max
GMM,1225.0,0.974719,0.014334,0.920630,0.965574,0.977148,0.985783,1.000000
HDBSCAN,1225.0,0.954342,0.050610,0.689372,0.960211,0.964906,0.968792,0.981119
K-Means,1225.0,0.975102,0.013626,0.929279,0.965983,0.977349,0.985599,0.998920


Bổ sung hai góc nhìn khác để kiểm chứng kết quả trên.

`reference_ari` so sánh kết quả của từng lần lặp với kết quả huấn luyện trên toàn
bộ 4.324 khách hàng, cho biết mô hình có tái tạo lại được phân cụm gốc hay không.

Riêng HDBSCAN, ARI còn được tính lại sau khi loại các điểm nhiễu để tách biệt hai
nguồn biến động: cấu trúc cụm thay đổi, hay chỉ ranh giới nhiễu dịch chuyển.

In [10]:
reference_scores = {
    model: reference_ari(X, label_matrices[model], model, configs[model])
    for model in configs
}

print(pd.DataFrame({
    model: {'mean': scores.mean(), 'std': scores.std(ddof=1), 'min': scores.min()}
    for model, scores in reference_scores.items()
}).T.to_string())

hdbscan_clean = pairwise_ari(label_matrices['HDBSCAN'], ignore_noise=True)
print(f"\nHDBSCAN sau khi loại điểm nhiễu: mean ARI = {hdbscan_clean.mean():.4f}, "
      f"std = {hdbscan_clean.std(ddof=1):.4f}")

             mean       std       min
GMM      0.981409  0.009605  0.958075
HDBSCAN  0.891814  0.036483  0.642442
K-Means  0.981069  0.009366  0.959902



HDBSCAN sau khi loại điểm nhiễu: mean ARI = 0.9953, std = 0.0231


## 5. Tổng hợp kết quả đánh giá

Bảng so sánh cuối cùng được lưu vào `data/processed/stability_results.csv`.
Cột `stability` được xếp theo cả Mean ARI và Std ARI: mức `Rất ổn định` yêu cầu
Mean ARI từ 0,90 và Std ARI không quá 0,03.

In [11]:
summary = summarize_stability(X, configs, label_matrices, iterations)
summary[['model', 'params', 'mean_ari', 'std_ari', 'min_ari', 'max_ari', 'stability']]

,model,params,mean_ari,std_ari,min_ari,max_ari,stability
2,K-Means,n_clusters=3,0.975102,0.013626,0.929279,0.998920,Rất ổn định
0,GMM,"n_components=3, covariance_type=spherical",0.974719,0.014334,0.920630,1.000000,Rất ổn định
1,HDBSCAN,"min_cluster_size=50, min_samples=None",0.954342,0.050610,0.689372,0.981119,Ổn định


In [12]:
summary.T

,2,0,1
model,K-Means,GMM,HDBSCAN
params,n_clusters=3,"n_components=3, covariance_type=spherical","min_cluster_size=50, min_samples=None"
n_iterations,50,50,50
n_pairs,1225,1225,1225
mean_ari,0.975102,0.974719,0.954342
std_ari,0.013626,0.014334,0.05061
min_ari,0.929279,0.92063,0.689372
max_ari,0.99892,1.0,0.981119
mean_reference_ari,0.981069,0.981409,0.891814
std_reference_ari,0.009366,0.009605,0.036483


In [13]:
iterations.to_csv('../outputs/results/stability_iterations.csv', index=False)
save_results(summary, '../data/processed/stability_results.csv')

Saved stability summary for 3 models to ../data/processed/stability_results.csv


,model,params,n_iterations,n_pairs,mean_ari,std_ari,min_ari,max_ari,mean_reference_ari,std_reference_ari,mean_n_clusters,min_n_clusters,max_n_clusters,mean_train_time,stability,mean_ari_no_noise,std_ari_no_noise,mean_noise_ratio
2,K-Means,n_clusters=3,50,1225,0.975102,0.013626,0.929279,0.998920,0.981069,0.009366,3.00,3,3,0.002888,Rất ổn định,NaN,NaN,NaN
0,GMM,"n_components=3, covariance_type=spherical",50,1225,0.974719,0.014334,0.920630,1.000000,0.981409,0.009605,3.00,3,3,0.005187,Rất ổn định,NaN,NaN,NaN
1,HDBSCAN,"min_cluster_size=50, min_samples=None",50,1225,0.954342,0.050610,0.689372,0.981119,0.891814,0.036483,3.02,3,4,0.036216,Ổn định,0.995279,0.023149,0.242752


## 6. Phân tích kết quả

**Thuật toán nào tạo ra các nhóm khách hàng nhất quán khi dữ liệu thay đổi nhỏ?**

K-Means (Mean ARI 0,9751) và GMM `spherical` (0,9747) đứng ngang nhau, chênh lệch
0,0004 nhỏ hơn nhiều so với Std ARI của cả hai (khoảng 0,014) nên không thể coi là
khác biệt thật. Cả hai đều cho đúng 3 cụm ở toàn bộ 50 lần lặp.

**Thuật toán nào nhạy cảm với sự thay đổi của dữ liệu?**

HDBSCAN. Mean ARI thấp hơn (0,9543), Std ARI cao gấp gần 4 lần (0,0506) và lần
lặp kém nhất chỉ đạt 0,6894. Số cụm cũng không cố định: một lần lặp cho 4 cụm thay
vì 3. Nguồn biến động nằm ở ranh giới nhiễu chứ không ở lõi cụm — sau khi loại các
điểm nhiễu, Mean ARI của HDBSCAN đạt 0,9953.

**Mô hình có chất lượng phân cụm cao có đồng thời ổn định hay không?**

Trong thực nghiệm này thì có. Thứ tự theo Silhouette ở Task 12 (GMM 0,4144 ≈
K-Means 0,4143 > HDBSCAN 0,2028) trùng với thứ tự theo Mean ARI. Không xuất hiện
trường hợp Silhouette cao đi kèm ARI thấp.

In [14]:
comparison = pd.read_csv('../outputs/results/clustering_experiments.csv')
best_quality = comparison.loc[comparison.groupby('model')['silhouette'].idxmax()]

(best_quality[['model', 'silhouette', 'davies_bouldin', 'calinski_harabasz']]
 .merge(summary[['model', 'mean_ari', 'std_ari', 'stability']], on='model')
 .sort_values('mean_ari', ascending=False))

,model,silhouette,davies_bouldin,calinski_harabasz,mean_ari,std_ari,stability
2,K-Means,0.414329,0.829263,4389.589669,0.975102,0.013626,Rất ổn định
0,GMM,0.414446,0.816665,4364.692454,0.974719,0.014334,Rất ổn định
1,HDBSCAN,0.202819,1.195553,2404.568174,0.954342,0.050610,Ổn định
